After running classify_scans.py:
1. Align all sources 
2. Detect duplicated scans 
3. Detect misaligned labeling

In [1]:
import pandas as pd
import os

# load all csv files in the current directory into a single DataFrame

# path to all csv files
path = "/home/gaia/Projects/legacy_data/manual_labeling/" # UPDATE PATH AS NEEDED
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
# save every df for debugging
dataframes = pd.DataFrame()
for f in csv_files: 
    df = pd.read_csv(os.path.join(path, f))
    df['csv_name'] = f
    dataframes = pd.concat([dataframes, df], ignore_index=True)
    dataframes['csv_name']

labeled_scans = dataframes



In [2]:
# amount of subject+session unique combination
unique_combinations = labeled_scans[['subject_id', 'session_id']].drop_duplicates()
print(f"Number of unique subject+session combinations: {len(unique_combinations)}")

Number of unique subject+session combinations: 1110


In [3]:
print(labeled_scans['csv_name'].unique())
print(f"scans per csv file: {labeled_scans.groupby('csv_name').size()}")

['BIDS_thirties_scan_classification_results.csv'
 'HardDrive_BIDS_over_thirties_scan_classification_results.csv'
 'gnbb_scan_classification_results.csv'
 'BIDS_under_thirties_scan_classification_results.csv'
 'AllTheRages_scan_classification_results.csv'
 'BIDS_over_thirties_scan_classification_results.csv'
 'drive_30to35_bids_scan_classification_results.csv']
scans per csv file: csv_name
AllTheRages_scan_classification_results.csv                      15
BIDS_over_thirties_scan_classification_results.csv                2
BIDS_thirties_scan_classification_results.csv                   240
BIDS_under_thirties_scan_classification_results.csv             489
HardDrive_BIDS_over_thirties_scan_classification_results.csv    279
drive_30to35_bids_scan_classification_results.csv                11
gnbb_scan_classification_results.csv                             90
dtype: int64


In [4]:
print(f"Total rows in combined DataFrame: {len(labeled_scans)}")

#unique scans = combination of subject_id and session_id
print(f"amount of unique scans: {len(labeled_scans[['subject_id', 'session_id']].drop_duplicates())}")

Total rows in combined DataFrame: 1126
amount of unique scans: 1110


In [5]:
# debug - can remove:
# is there a subject_id that doesn't start with sub- ? 
subject_ids = labeled_scans['subject_id'].unique()
non_sub_ids = [sid for sid in subject_ids if not sid.startswith('sub-')]
print(f"Found {len(non_sub_ids)} subject IDs that don't start with 'sub-': {non_sub_ids}")

# is there a session_id that doesn't start with ses- ? 
session_ids = labeled_scans['session_id'].unique()
non_ses_ids = [sid for sid in session_ids if not sid.startswith('ses-')]
print(f"Found {len(non_ses_ids)} session IDs that don't start with 'ses-': {non_ses_ids}")


Found 0 subject IDs that don't start with 'sub-': []
Found 0 session IDs that don't start with 'ses-': []


In [6]:
# handling duplicated labels:

duplicates = labeled_scans[labeled_scans.duplicated(subset=['subject_id', 'session_id'], keep=False)]

non_duplicated = labeled_scans[~labeled_scans.duplicated(subset=['subject_id', 'session_id'], keep=False)]
print(f"Number of non-duplicated rows: {len(non_duplicated)}")

# order by subject_id and session_id for easier inspection
duplicates = duplicates.sort_values(by=['subject_id', 'session_id'])
print(f"Number of duplicate rows based on subject_id and session_id: {len(duplicates)}")

# for every duplicated subject_id and session_id, check if classification_label is the same
conflicting_labels = duplicates.groupby(['subject_id', 'session_id']).filter(lambda x: len(x['classification_label'].unique()) > 1)
print(f"Number of rows with conflicting labels for the same subject_id and session_id: {len(conflicting_labels)}")



Number of non-duplicated rows: 1094
Number of duplicate rows based on subject_id and session_id: 32
Number of rows with conflicting labels for the same subject_id and session_id: 22


In [7]:
non_conflicting_duplicates = duplicates.groupby(['subject_id', 'session_id']).filter(lambda x: len(x['classification_label'].unique()) == 1)
print(f"Number of non-conflicting duplicates: {len(non_conflicting_duplicates)}")

non_conflicting_duplicates_unique_scans = non_conflicting_duplicates[['subject_id', 'session_id', 'classification_label']].drop_duplicates()
print(f"number of non-conflicting duplicated unique scans: {len(non_conflicting_duplicates_unique_scans)}")

Number of non-conflicting duplicates: 10
number of non-conflicting duplicated unique scans: 5


go over "conflicting labels" and manually and align 

### temporary resolution of conflicts 

In [8]:
# temporary - for each scan (subject_id, session_id), get the best classification_label

# 1. Calculate the minimal label for each group and assign it to a new column
# This keeps all original columns and rows intact
conflicting_labels['classification_label'] = conflicting_labels.groupby(['subject_id', 'session_id'])['classification_label'].transform('min')

# 2. If you want to keep only one row per scan (now that they have the same label)
# but keep all 5 columns of information:
resolved_labels = conflicting_labels.drop_duplicates(subset=['subject_id', 'session_id'])


print(f"number of resolved unique scans: {len(resolved_labels)}")



number of resolved unique scans: 11


In [9]:
# concatinate resolved_labels and non_conflicting_duplicates_unique_scans
resolved_duplications_unique_scans = pd.concat([resolved_labels, non_conflicting_duplicates_unique_scans], ignore_index=True)
print(f"Number of resolved duplication unique scans: {len(resolved_duplications_unique_scans)}")


Number of resolved duplication unique scans: 16


In [10]:
# add the resolved_duplications to non_duplicated


final_labeled_scans = pd.concat([non_duplicated, resolved_duplications_unique_scans], ignore_index=True)
print(f"Number of final labeled scans: {len(final_labeled_scans)}")



Number of final labeled scans: 1110


In [11]:
final_labeled_scans.to_csv("/home/gaia/Projects/legacy_data/legacy_pipe/data/interim/manual_labels_resolved_duplicates.csv", index=False)

In [12]:
# add the manual label column to combined_df by matching the subject_id and session_id
combined_df = pd.read_pickle('/home/gaia/Projects/legacy_data/combined_gm_volumes.pkl')
print(combined_df.columns)


Index(['subject_id', 'session_id', 'region_label', 'tissue_type', 'volume_mm3',
       'tiv', 'sex', 'institute', 'manufacturer', 'age_in_years', 'dob',
       'gm_volume_cm3', 'protocol', 'source', 'birth_year', 'unnamed: 0',
       'atlas_name', 'scan_time', 'age_at_scan', 'weight', 'directory_path',
       'estimated_critical_info', 'scan_date', 'file_path', 'scan_year'],
      dtype='object')


In [13]:
# remove 'file_path' column from labeled_scans 
labeled_scans_for_merge = labeled_scans[['subject_id', 'session_id', 'classification_label']]

# remove "sub-" prefix from subject_id and "ses-" prefix from session_id
labeled_scans_for_merge['subject_id'] = labeled_scans_for_merge['subject_id'].str.replace('sub-', '', regex=False)
labeled_scans_for_merge['session_id'] = labeled_scans_for_merge['session_id'].str.replace('ses-', '', regex=False)

combined_df['subject_id'] = combined_df['subject_id'].str.replace('sub-', '', regex=False)
combined_df['session_id'] = combined_df['session_id'].str.replace('ses-', '', regex=False)

/tmp/ipykernel_94750/1792195024.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  labeled_scans_for_merge['subject_id'] = labeled_scans_for_merge['subject_id'].str.replace('sub-', '', regex=False)
/tmp/ipykernel_94750/1792195024.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  labeled_scans_for_merge['session_id'] = labeled_scans_for_merge['session_id'].str.replace('ses-', '', regex=False)


In [14]:
# convert the str yyyy-mm-dd to str yyyymmdd
combined_df['scan_date'] = combined_df['scan_date'].str.replace('-', '', regex=False)

# check if the column "classification_label" exists before merging. if yes, update existing column. if not - add new column
if 'classification_label' in combined_df.columns:
    combined_df = combined_df.drop(columns=['classification_label'])
combined_df = combined_df.merge(
    labeled_scans_for_merge.rename(columns={'session_id': 'scan_date'}), 
    on=['subject_id', 'scan_date'], 
    how='left'
)
print(combined_df.columns)


Index(['subject_id', 'session_id', 'region_label', 'tissue_type', 'volume_mm3',
       'tiv', 'sex', 'institute', 'manufacturer', 'age_in_years', 'dob',
       'gm_volume_cm3', 'protocol', 'source', 'birth_year', 'unnamed: 0',
       'atlas_name', 'scan_time', 'age_at_scan', 'weight', 'directory_path',
       'estimated_critical_info', 'scan_date', 'file_path', 'scan_year',
       'classification_label'],
      dtype='object')


In [15]:
print(f"unique classification_label in combined_df: {combined_df['classification_label'].unique()}")

unique classification_label in combined_df: [ 1.  5. nan  4.  2.  3.]


In [16]:
# keep only classification_label=1 and snbb
best_combined_df = combined_df[(combined_df['classification_label'] == 1) | (combined_df['source'] == 'snbb')]

best_combined_df.to_pickle('/home/gaia/Projects/legacy_data/best_combined_gm_volumes.pkl')
